# BirdCLEF 2026 - Submission v41 - Ensemble + Strong TTA

Combines two models:
1. **exp78** - the model that scored 0.592 on the LB (head-only fine-tune on focal recordings).
2. **v40_strong** - new model retrained with: stratified focal sampling + ALL train_soundscapes mixed in + mixup + last-30-layer fine-tune. Trained to maximise *macro-AUC on held-out soundscape files*, the metric that matches the LB.

TTA: 5 temporal offsets + 2 light SpecAugment variants per row, geometric-mean aggregation.

Both models use the same input shape (64, 626, 3) and the same exp78 mel-spec preprocessing, so a single forward pipeline serves both.

In [ ]:
import os, glob, json, shutil, zipfile, warnings
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
from tensorflow import keras

warnings.filterwarnings('ignore')
print('TF', tf.__version__, ' Librosa', librosa.__version__)

In [ ]:
# =======================================================
# PATHS  (3-model ensemble: exp78 + v40_strong + v40b_strong)
# =======================================================
MODEL_EXP78 = '/kaggle/input/datasets/danielemalerba0302/birdclef2026-model/exp78_full_model.keras'
MODEL_V40   = '/kaggle/input/datasets/danielemalerba0302/birdclef2026-model/model_v40_strong.keras'
MODEL_V40B  = '/kaggle/input/datasets/danielemalerba0302/birdclef2026-model/model_v40b_strong.keras'
TEST_AUDIO_DIR = '/kaggle/input/competitions/birdclef-2026/test_soundscapes'
SAMPLE_SUB_PATH = '/kaggle/input/competitions/birdclef-2026/sample_submission.csv'
SUBMISSION_PATH = '/kaggle/working/submission.csv'

# Ensemble weights (geometric mean) chosen by 3-way sweep on cached
# soundscape val:
#   exp78 alone  = 0.6004
#   v40   alone  = 0.6261
#   v40b  alone  = 0.5785  (weaker alone, but adds diversity)
#   greedy clean = 0.017 exp78 + 0.017 v40 + 0.966 v40b -> 0.6632
#   safe 2-way   = 0.20 exp78 + 0.80 v40            -> 0.6325
# Soundscape val tracks the LB tightly (exp78 local 0.6004 vs LB 0.592).
W_EXP78 = 0.017
W_V40   = 0.017
W_V40B  = 0.966

# =======================================================
# AUDIO PARAMETERS - MUST match exp78 / v40 training
# =======================================================
SAMPLE_RATE = 32000
DURATION = 5.0
N_MELS = 64
N_FFT = 2048
HOP_LENGTH = 256
FMIN = 20
FMAX = 16000
TOP_DB = 40.0
MEL_NORM = 'slaney'
USE_HTK = True
TARGET_HEIGHT = 64
TARGET_WIDTH = 626

# TTA
TTA_OFFSETS = [-1.0, -0.5, 0.0, 0.5, 1.0]
N_SPECAUG_TTA = 2
USE_GEOMETRIC_MEAN = True
BATCH_SIZE = 32

exp78_present = os.path.exists(MODEL_EXP78)
v40_present   = os.path.exists(MODEL_V40)
v40b_present  = os.path.exists(MODEL_V40B)
print('exp78 model present:', exp78_present)
print('v40   model present:', v40_present)
print('v40b  model present:', v40b_present)
assert exp78_present or v40_present or v40b_present, 'no model available'
if not exp78_present:
    W_EXP78 = 0.0
if not v40_present:
    W_V40 = 0.0
if not v40b_present:
    W_V40B = 0.0
total_w = W_EXP78 + W_V40 + W_V40B
assert total_w > 0, 'all weights are zero'
print(f'weights: exp78={W_EXP78:.2f} v40={W_V40:.2f} v40b={W_V40B:.2f}')


In [ ]:
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
SPECIES_LIST = list(sample_sub.columns[1:])
assert len(SPECIES_LIST) == 234

In [ ]:
# =======================================================
# .keras patcher (handles BatchNorm renorm keys for Keras compat)
# =======================================================
BAD_KEYS = ['renorm', 'renorm_clipping', 'renorm_momentum', 'quantization_config']

def _strip_keys(obj):
    if isinstance(obj, dict):
        for k in BAD_KEYS:
            obj.pop(k, None)
        for v in obj.values():
            _strip_keys(v)
    elif isinstance(obj, list):
        for it in obj:
            _strip_keys(it)

def patch_keras(src_path, tag):
    out_path = f'/kaggle/working/{tag}_patched.keras'
    tmp = f'/kaggle/working/{tag}_patch_tmp'
    if os.path.exists(tmp):
        shutil.rmtree(tmp)
    os.makedirs(tmp)
    with zipfile.ZipFile(src_path, 'r') as z:
        z.extractall(tmp)
    cfg_path = os.path.join(tmp, 'config.json')
    with open(cfg_path) as f:
        cfg = json.load(f)
    _strip_keys(cfg)
    with open(cfg_path, 'w') as f:
        json.dump(cfg, f)
    with zipfile.ZipFile(out_path, 'w', zipfile.ZIP_DEFLATED) as z:
        for root, _, files in os.walk(tmp):
            for fn in files:
                full = os.path.join(root, fn)
                z.write(full, os.path.relpath(full, tmp))
    return out_path

models = []
weights_list = []

if exp78_present and W_EXP78 > 0:
    p = patch_keras(MODEL_EXP78, 'exp78')
    m = keras.models.load_model(p, compile=False, safe_mode=False)
    print('exp78 input', m.input_shape, ' output', m.output_shape)
    assert m.input_shape[1:] == (TARGET_HEIGHT, TARGET_WIDTH, 3)
    assert m.output_shape[-1] == 234
    models.append(('exp78', m))
    weights_list.append(W_EXP78)

if v40_present and W_V40 > 0:
    p = patch_keras(MODEL_V40, 'v40')
    m = keras.models.load_model(p, compile=False, safe_mode=False)
    print('v40   input', m.input_shape, ' output', m.output_shape)
    assert m.input_shape[1:] == (TARGET_HEIGHT, TARGET_WIDTH, 3)
    assert m.output_shape[-1] == 234
    models.append(('v40', m))
    weights_list.append(W_V40)

if v40b_present and W_V40B > 0:
    p = patch_keras(MODEL_V40B, 'v40b')
    m = keras.models.load_model(p, compile=False, safe_mode=False)
    print('v40b  input', m.input_shape, ' output', m.output_shape)
    assert m.input_shape[1:] == (TARGET_HEIGHT, TARGET_WIDTH, 3)
    assert m.output_shape[-1] == 234
    models.append(('v40b', m))
    weights_list.append(W_V40B)

import numpy as _np
weights = _np.array(weights_list, dtype=_np.float32)
weights = weights / weights.sum()
print('Active models:', [n for n, _ in models])
print('Normalised weights:', weights.tolist())


In [ ]:
# =======================================================
# AUDIO -> SPECTROGRAM (exp78 pipeline)
# =======================================================

def audio_to_spectrogram(audio_arr, sr=SAMPLE_RATE):
    mel = librosa.feature.melspectrogram(
        y=audio_arr, sr=sr,
        n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH,
        fmin=FMIN, fmax=FMAX, norm=MEL_NORM, htk=USE_HTK,
    )
    mel = np.nan_to_num(mel, nan=0.0, posinf=0.0, neginf=0.0)
    mel_db = librosa.power_to_db(mel, ref=np.max, top_db=TOP_DB)
    mel_db = np.nan_to_num(mel_db, nan=-TOP_DB, posinf=0.0, neginf=-TOP_DB)
    mel_norm = (mel_db + TOP_DB) / TOP_DB
    mel_norm = np.clip(mel_norm, 0, 1)
    spec = np.stack([mel_norm, mel_norm, mel_norm], axis=-1).astype(np.float32)
    if spec.shape[1] < TARGET_WIDTH:
        spec = np.pad(spec, ((0, 0), (0, TARGET_WIDTH - spec.shape[1]), (0, 0)), mode='constant')
    elif spec.shape[1] > TARGET_WIDTH:
        spec = spec[:, :TARGET_WIDTH, :]
    return spec

def light_specaugment(spec, rng):
    spec = spec.copy()
    H, W, _ = spec.shape
    fw = rng.randint(2, 8)
    fs = rng.randint(0, max(1, H - fw))
    spec[fs:fs+fw, :, :] = 0.0
    tw = rng.randint(5, 25)
    ts = rng.randint(0, max(1, W - tw))
    spec[:, ts:ts+tw, :] = 0.0
    return spec

def extract_shifted_block(y, start_second, offset_second):
    block_samples = int(DURATION * SAMPLE_RATE)
    start_sample = int(round((start_second + offset_second) * SAMPLE_RATE))
    end_sample = start_sample + block_samples
    left_pad = max(0, -start_sample)
    right_pad = max(0, end_sample - len(y))
    start_sample = max(0, start_sample)
    end_sample = min(len(y), end_sample)
    block = y[start_sample:end_sample]
    if left_pad or right_pad:
        block = np.pad(block, (left_pad, right_pad), mode='constant')
    if len(block) < block_samples:
        block = np.pad(block, (0, block_samples - len(block)), mode='constant')
    elif len(block) > block_samples:
        block = block[:block_samples]
    return block

def build_tta_specs_for_row(y, end_second, rng):
    start_second = end_second - int(DURATION)
    out = []
    for off in TTA_OFFSETS:
        block = extract_shifted_block(y, start_second, off)
        out.append(audio_to_spectrogram(block, sr=SAMPLE_RATE))
    centered = audio_to_spectrogram(extract_shifted_block(y, start_second, 0.0), sr=SAMPLE_RATE)
    for _ in range(N_SPECAUG_TTA):
        out.append(light_specaugment(centered, rng))
    return out

# Sanity
dummy = np.zeros(int(SAMPLE_RATE * DURATION), dtype=np.float32)
test_spec = audio_to_spectrogram(dummy)
for name, m in models:
    assert test_spec.shape == m.input_shape[1:], f'{name} shape mismatch'
print('shape sanity OK')

In [ ]:
# =======================================================
# AGGREGATION
# =======================================================
_LOG_EPS = 1e-7

def aggregate_tta_per_model(pred_list):
    arr = np.stack(pred_list, axis=0)
    if USE_GEOMETRIC_MEAN:
        return np.exp(np.log(np.clip(arr, _LOG_EPS, 1.0)).mean(axis=0))
    return arr.mean(axis=0)

def ensemble(per_model_aggs):
    """per_model_aggs is a list aligned to `models`/`weights`.
    Each entry is a (n_classes,) numpy array."""
    stack = np.stack(per_model_aggs, axis=0)
    if USE_GEOMETRIC_MEAN:
        log_stack = np.log(np.clip(stack, _LOG_EPS, 1.0))
        agg = np.exp((weights[:, None] * log_stack).sum(axis=0))
    else:
        agg = (weights[:, None] * stack).sum(axis=0)
    return np.clip(agg, 0.0, 1.0)

In [ ]:
# =======================================================
# INFERENCE
# Per row: build TTA specs once, predict with each model, aggregate.
# =======================================================
audio_files = []
for ext in ['*.ogg', '*.wav', '*.flac', '*.mp3']:
    audio_files.extend(glob.glob(os.path.join(TEST_AUDIO_DIR, '**', ext), recursive=True))
audio_files = sorted(audio_files)
print('Test audio files:', len(audio_files))

expected_by_file = {}
for row_id in sample_sub['row_id']:
    file_stem = '_'.join(row_id.split('_')[:-1])
    expected_by_file.setdefault(file_stem, []).append(row_id)

# Per (row_id, model_name) -> list of TTA preds
preds_by_row_model = {}  # {(row_id, name): [pred, pred, ...]}

rng = np.random.RandomState(20260507)

# To minimise model-switch overhead, process per file: build all TTA specs,
# then run each model over the whole batch, then aggregate.
for i, audio_path in enumerate(audio_files):
    file_stem = os.path.splitext(os.path.basename(audio_path))[0]
    if i % 5 == 0:
        print(f'{i+1}/{len(audio_files)}: {file_stem}')
    if file_stem not in expected_by_file:
        continue
    y, _ = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)

    file_specs = []
    file_row_ids = []  # parallel list, one per spec
    for row_id in expected_by_file[file_stem]:
        end_sec = int(row_id.split('_')[-1])
        for spec in build_tta_specs_for_row(y, end_sec, rng):
            file_specs.append(spec)
            file_row_ids.append(row_id)
    if not file_specs:
        continue
    X = np.array(file_specs, dtype=np.float32)

    for name, m in models:
        preds = m.predict(X, batch_size=BATCH_SIZE, verbose=0)
        preds = np.clip(preds, 0.0, 1.0)
        for rid, p in zip(file_row_ids, preds):
            preds_by_row_model.setdefault((rid, name), []).append(p)

print('rows x models with predictions:', len(preds_by_row_model))

In [ ]:
# =======================================================
# BUILD SUBMISSION
# =======================================================
expected_row_ids = list(sample_sub['row_id'])
model_names = [n for n, _ in models]

if not preds_by_row_model:
    print('No predictions -> placeholder zeros (visible-run fallback)')
    arr = np.zeros((len(expected_row_ids), len(SPECIES_LIST)), dtype=np.float32)
else:
    rows = []
    for rid in expected_row_ids:
        per_model = []
        for name in model_names:
            key = (rid, name)
            if key in preds_by_row_model:
                per_model.append(aggregate_tta_per_model(preds_by_row_model[key]))
            else:
                per_model.append(np.zeros(len(SPECIES_LIST), dtype=np.float32))
        rows.append(ensemble(per_model))
    arr = np.stack(rows, axis=0).astype(np.float32)

submission_df = pd.DataFrame(arr, columns=SPECIES_LIST)
submission_df.insert(0, 'row_id', expected_row_ids)
submission_df.to_csv(SUBMISSION_PATH, index=False)
print('Saved', SUBMISSION_PATH, 'shape', submission_df.shape)
submission_df.head()

In [ ]:
# Verification
check_df = pd.read_csv(SUBMISSION_PATH)
sample_check = pd.read_csv(SAMPLE_SUB_PATH)
assert list(check_df.columns) == list(sample_check.columns)
assert list(check_df['row_id']) == list(sample_check['row_id'])
vals = check_df.iloc[:, 1:].values
print('rows', len(check_df), ' cols', len(check_df.columns))
print('min', vals.min(), ' max', vals.max(), ' nan', np.isnan(vals).any())
assert not np.isnan(vals).any()
assert vals.min() >= 0 and vals.max() <= 1
print('Submission ready.')